# 06 章 / 第 3 节 · GRPO + verifiable reward(对齐 DeepSeek-R1 路线)

## 学习目标
- 用 **TRL GRPOTrainer** + **可验证奖励**(规则化 reward)而非偏好对,在 Qwen2.5-1.5B 上做数学推理对齐
- 理解 GRPO vs PPO 的核心差异:**group baseline 替代 value model**,显存省一半
- 手写两个 reward function:`format_reward`(<think> 结构)+ `answer_reward`(答案精确匹配)
- 看 `<think>...</think>` 在训练中**涌现**(可以理解为 reasoning 的雏形)

## 对应八股
`liguodongiot/llm-action`:**`llm-interview/llm-rlhf.md`**(GRPO 节)+ `llm-alignment/`

## ⚠️ 运行要求
**Linux + CUDA GPU 12GB+**(TRL GRPOTrainer 在 Windows 也能跑,但显存占用大、慢)。
本节用 `Qwen2.5-1.5B-Instruct` + `num_generations=4`,12GB 卡 +`gradient_checkpointing` 刚好。


## 1. 概念背景

### 1.1 为什么不再用偏好对 + DPO?

传统 RLHF / DPO 流派要 **人工标注 chosen vs rejected**,问题:
- 数据贵(标 1k 对偏好 ≈ $1000+)
- 标注员对**推理过程**的评分很不一致
- Reward model(若走 PPO)自己就是偏差源

### 1.2 Verifiable Reward 的洞察

**如果任务本身有 ground truth**(数学题答案、代码 test case、逻辑题答案),
就**根本不需要训 reward model**:对了 +1,错了 0,format 错 -0.5。
DeepSeek-R1-Zero(2025-01)、Open-R1、nanochat(2025-10)都走这条路。

### 1.3 GRPO 是什么

GRPO = **G**roup **R**elative **P**olicy **O**ptimization(DeepSeekMath 论文 2402.03300)。

核心改动:
- **PPO**:`advantage = reward - V(s)`,需要单独训 value network
- **GRPO**:对同一个 prompt 采样 `G` 条 response,**advantage = (r_i - mean(r_1..G)) / std(r_1..G)**
  —— 用 group baseline 替代 value model,**省掉一个模型的显存**

实现上 = PPO loss + 几个 sampler/advantage 改动。代码差异不大,思路飞跃。

### 1.4 本节流程

```
Qwen2.5-1.5B-Instruct
  ↓ GSM8K-zh 题目 + system prompt(要求 <think>...</think><answer>...</answer> 格式)
  ↓ 对每个 prompt 采样 4 条 completion
  ↓ format_reward + answer_reward 打分
  ↓ GRPO 用 group baseline 更新 policy
  ↓ 200 步后,看 reward 曲线 + 推理样本
```


## 2. 环境检查


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
from scripts.env_check import check
check()


## 3. 加载模型与 tokenizer


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float32

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype = dtype,
    device_map = device,
)
print(f"模型加载完成 | dtype: {dtype} | device: {device}")
if device == "cuda":
    print(f"显存占用: {torch.cuda.memory_allocated()/1e9:.2f} GB")


## 4. 数据:GSM8K-zh 中文数学题


In [ ]:
from datasets import load_dataset

# meta-math/GSM8K_zh 是 GSM8K 的中文翻译版,约 8k train / 1.3k test
ds = load_dataset("meta-math/GSM8K_zh", split="train")
print(f"训练样本: {len(ds)}")
print(f"字段: {ds.column_names}")
print(f"\n--- 一条样本 ---")
print(f"question_zh:\n{ds[0]['question_zh']}")
print(f"\nanswer_only:\n{ds[0]['answer_only']}")


## 5. 构造训练样本(prompt + ground truth)

GRPO 训练需要 `prompt` 字段;reward function 通过额外字段拿 ground truth。


In [ ]:
SYSTEM_PROMPT = """你是一个善于推理的数学助手。请严格按照以下格式回答:
<think>
在这里写出详细的推理步骤
</think>
<answer>
在这里写出最终的数字答案,只写一个数字
</answer>"""

def make_prompt(q_zh: str) -> list[dict]:
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": q_zh},
    ]

def prepare(example):
    return {
        "prompt": make_prompt(example["question_zh"]),
        "ground_truth": str(example["answer_only"]).strip(),
    }

# 子集训练(教学跑,真复现用 ds.select(range(len(ds))))
SUBSET = 500
train_ds = ds.shuffle(seed=42).select(range(SUBSET)).map(prepare, remove_columns=ds.column_names)
print(f"准备好的训练样本: {len(train_ds)}")
print(f"\n--- 示例 prompt ---")
for msg in train_ds[0]['prompt']:
    print(f"[{msg['role']}] {msg['content'][:120]}...")
print(f"\n[ground_truth] {train_ds[0]['ground_truth']}")


## 6. Reward 函数 ⭐⭐(本节核心)

两个 reward 函数:
1. **`format_reward`** — 看 completion 是否含 `<think>...</think><answer>...</answer>` 结构;有 +0.5,无 0
2. **`answer_reward`** — 从 `<answer>` 标签里提取数字,与 ground truth 精确匹配;对 +1.5,错 0

权重设计:answer 是主要信号(1.5),format 是辅助(0.5);
**总和封顶 2.0** 让 reward 曲线易读。

> **踩坑提醒**:format reward 权重不能太高,否则模型学会"刷格式但乱答",
> 这是 reward hacking 经典症状。


In [ ]:
import re

THINK_RE   = re.compile(r"<think>(.*?)</think>", re.DOTALL)
ANSWER_RE  = re.compile(r"<answer>(.*?)</answer>", re.DOTALL)
NUMBER_RE  = re.compile(r"-?\d+(?:\.\d+)?")


def extract_answer(text: str) -> str | None:
    """从 completion 文本里抠出 <answer> 标签里的数字。"""
    m = ANSWER_RE.search(text)
    if not m:
        return None
    nums = NUMBER_RE.findall(m.group(1))
    return nums[-1] if nums else None  # 取最后一个数字(防多 candidate)


def format_reward(completions, **kwargs) -> list[float]:
    """TRL GRPO 约定:输入 list[list[dict]](每条 = 一段对话),输出 list[float]。"""
    rewards = []
    for completion in completions:
        # completion 是 [{"role": "assistant", "content": "..."}]
        text = completion[0]["content"]
        has_think  = bool(THINK_RE.search(text))
        has_answer = bool(ANSWER_RE.search(text))
        rewards.append(0.5 if (has_think and has_answer) else 0.0)
    return rewards


def answer_reward(completions, ground_truth=None, **kwargs) -> list[float]:
    """完全精确匹配 ground truth(数字)。"""
    rewards = []
    for completion, gt in zip(completions, ground_truth):
        text = completion[0]["content"]
        pred = extract_answer(text)
        if pred is None:
            rewards.append(0.0)
            continue
        # 转 float 比较,容忍 "5.0" vs "5"
        try:
            ok = abs(float(pred) - float(gt)) < 1e-3
        except ValueError:
            ok = False
        rewards.append(1.5 if ok else 0.0)
    return rewards


# 单测
fake_completions = [
    [{"role": "assistant", "content": "<think>3+5=8</think><answer>8</answer>"}],   # 全对
    [{"role": "assistant", "content": "<think>guessing</think><answer>9</answer>"}], # format 对,答案错
    [{"role": "assistant", "content": "is 8"}],                                       # 全没有
]
fake_gt = ["8", "8", "8"]
print("format rewards:", format_reward(fake_completions))
print("answer rewards:", answer_reward(fake_completions, ground_truth=fake_gt))


## 7. GRPO 配置


In [ ]:
from trl import GRPOConfig, GRPOTrainer

OUTPUT_DIR = "../checkpoints/06_grpo_verifiable"

cfg = GRPOConfig(
    output_dir = OUTPUT_DIR,

    # ---- 训练循环 ----
    per_device_train_batch_size = 1,            # GRPO 显存吃在 num_generations 上
    gradient_accumulation_steps = 4,
    num_generations = 4,                        # ⭐ 每个 prompt 采样 4 条,group baseline 用
    max_completion_length = 256,                # 别给太长,显存会炸
    max_prompt_length = 512,

    # ---- 学习率 ----
    learning_rate = 5e-6,                       # 比 SFT 的 2e-4 低 40x
    lr_scheduler_type = "cosine",
    warmup_steps = 10,
    max_steps = 200,
    optim = "adamw_torch",                      # GRPO 推荐 adamw_torch 而非 8bit
    weight_decay = 0.01,
    beta = 0.04,                                # KL 系数,从 0.04 起步

    # ---- 采样 ----
    temperature = 0.9,
    top_p = 0.95,

    # ---- 系统 ----
    bf16 = True,
    gradient_checkpointing = True,
    report_to = "none",
    logging_steps = 5,
    save_steps = 100,
    save_total_limit = 2,
    seed = 42,

    # ---- (可选)vLLM 加速采样,Linux 限定 ----
    # use_vllm = True,
    # vllm_device = "cuda:0",
)

trainer = GRPOTrainer(
    model = model,
    args = cfg,
    train_dataset = train_ds,
    reward_funcs = [format_reward, answer_reward],   # 多个 reward 自动求和
    processing_class = tokenizer,
)


## 8. 跑训练


In [ ]:
import time

t0 = time.time()
stats = trainer.train()
elapsed = time.time() - t0

print(f"\n=== 训练完成 ===")
print(f"  总耗时: {elapsed/60:.1f} min")
print(f"  末步 loss: {stats.training_loss:.4f}")
if device == "cuda":
    peak = torch.cuda.max_memory_allocated() / 1e9
    print(f"  显存峰值: {peak:.2f} GB")


## 9. 看 reward 曲线 + format/answer 分项


In [ ]:
import matplotlib.pyplot as plt

# GRPOTrainer 把每个 reward function 的得分单独 log
hist = trainer.state.log_history
def get_series(key):
    return [(l["step"], l[key]) for l in hist if key in l]

reward_total  = get_series("reward")
reward_fmt    = get_series("rewards/format_reward")
reward_ans    = get_series("rewards/answer_reward")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
if reward_total:
    s, r = zip(*reward_total); ax[0].plot(s, r, label="total")
if reward_fmt:
    s, r = zip(*reward_fmt);   ax[0].plot(s, r, label="format", alpha=0.7)
if reward_ans:
    s, r = zip(*reward_ans);   ax[0].plot(s, r, label="answer", alpha=0.7)
ax[0].set_title("GRPO reward curves"); ax[0].set_xlabel("step")
ax[0].legend(); ax[0].grid(alpha=0.3)

# loss
losses = get_series("loss")
if losses:
    s, l = zip(*losses); ax[1].plot(s, l); ax[1].set_title("policy loss")
    ax[1].set_xlabel("step"); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()


## 10. 推理样本:看 `<think>` 是否涌现


In [ ]:
model.eval()

# 拿测试集试试(没训过的样本)
test = load_dataset("meta-math/GSM8K_zh", split="test").select(range(3))

for ex in test:
    prompt = make_prompt(ex["question_zh"])
    text = tokenizer.apply_chat_template(prompt, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(device)
    out = model.generate(**inputs, max_new_tokens=400, do_sample=True,
                          temperature=0.7, top_p=0.9)
    response = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    print(f"=== 题目 ===\n{ex['question_zh']}")
    print(f"\n=== Ground Truth ===\n{ex['answer_only']}")
    print(f"\n=== 模型回答 ===\n{response}")
    print("=" * 60)


## 11. 踩坑记录

- **`num_generations` 必须 ≥ 2,推荐 4-8**:GRPO 的 group baseline 至少要 2 条样本算 mean / std。`num_generations=4` 在 12GB 卡上是教学甜点。
- **`max_completion_length` 千万别开太大**:`num_generations × batch × max_completion_length` 占显存,256 是 12GB 卡的安全值。给 1024 直接 OOM。
- **`learning_rate=5e-6` 是经验起步,千万不要照搬 SFT 的 2e-4**:GRPO 对 lr 极敏感,大了 reward 直接崩溃,小了几百步看不到提升。
- **`beta`(KL 系数)的取舍**:大(0.1+)= policy 不敢偏离原模型,reward 涨不动;小(0.01-)= 偏离 ref 太多,推理质量退化。`0.04` 是 R1 / Open-R1 经验值。
- **reward hacking 的两种症状**:
  1. 模型只输出 `<think></think><answer>0</answer>`(刷 format 但乱答)→ 降 format 权重 / 加格式细则惩罚
  2. 模型只复读题目里的数字 → answer reward 加 "和题目数字不能完全相同" 约束
- **vLLM 加速生成**:Linux + Qwen2.5-1.5B 在 vLLM 上采样速度比 transformers 快 5-10x,真训记得开 `use_vllm=True`。
- **格式 reward 不要给太多**:可以从 0.5 起,看 format 学得快就降到 0.1;否则模型很快"满级 format"就不再学解题。

## 12. 延伸阅读

- 论文:`/paper fetch 2402.03300` (DeepSeekMath / GRPO)
- 论文:`/paper fetch 2501.12948` (DeepSeek-R1)
- [TRL GRPOTrainer 文档](https://huggingface.co/docs/trl/grpo_trainer)
- [Open-R1 完整复现](https://github.com/huggingface/open-r1)
- [Phil Schmid mini-R1](https://www.philschmid.de/mini-deepseek-r1)
- [[Slipbox/grpo-vs-ppo]]
- [[Slipbox/verifiable-reward-design]]
- 本仓 Capstone:`09_capstone/mini_r1_gsm8k_zh.ipynb`(下一步)
